# Switch-point evaluation — LogReg vs BiLSTM

**Why this notebook exists.** The token-level L1/L2 metrics in the main tutorial
(acc ≈ 0.98, AUC ≈ 0.999) are heavily flattered by the class imbalance at the
token level *and* by the fact that language changes only on ~1.8% of positions.
98% of the time the language just continues — so a trivial "same as last word"
baseline already looks great.

The real linguistic question is: **can the model find the switch points?**
We redefine the target as

> `switch_t = 1  if  lang_t ≠ lang_{t-1}  (within the same sentence), else 0`

and evaluate both models on this task directly.

We reuse the same dialogue-level 80/10/10 split and the same two models from
`code_switching_tutorial.ipynb`.


In [ ]:
import random, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, average_precision_score, f1_score, accuracy_score,
)
from scipy.sparse import hstack, csr_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CORPUS = Path("../Data/CS Corpus Prediction/BangorCorpus.txt")
assert CORPUS.exists(), CORPUS

## 1. Load + tokenize to word-level rows

Identical to §2–§3 of the tutorial.


In [ ]:
df = pd.read_csv(CORPUS, sep="\t")
TOP_LANGS = ("eng", "spa")

def tokenize(utt, syn):
    if not isinstance(utt, str) or not isinstance(syn, str): return None
    toks, tags = utt.split(","), syn.split(".")
    if len(toks) != len(tags): return None
    out = []
    for tok, pos in zip(toks, tags):
        if "." not in tok: return None
        w, l = tok.rsplit(".", 1)
        out.append((w.strip(), l.strip(), pos.strip()))
    return out

def ok(triples, allowed=("eng","spa","amb")):
    return all(l in allowed for _, l, _ in triples)

rows = []
dialogue_to_id = {sf: i for i, sf in enumerate(sorted(df["Soundfile"].unique()))}
sent_cnt = defaultdict(int)
for _, r in df.iterrows():
    t = tokenize(r["Utterance"], r["Syntax"])
    if t is None or not ok(t):
        continue
    did = dialogue_to_id[r["Soundfile"]]
    sent_cnt[did] += 1
    sid = sent_cnt[did]
    for wi, (w, l, p) in enumerate(t, 1):
        rows.append({"dialogue_id": did, "sentence_id": sid,
                     "speaker": r["Speaker"], "word_index": wi,
                     "word": w, "word_lang": l, "pos_tag": p})

tidy = pd.DataFrame(rows)
print("Tidy rows:", f"{len(tidy):,}")

## 2. Framing + dialogue-level split

Drop `amb`, label L2 = spa = 1. Compute the **true** switch label
`switch_true` on every token (NaN on the first word of every sentence because
there is no previous word to compare against).


In [ ]:
data = tidy[tidy["word_lang"].isin(TOP_LANGS)].copy().reset_index(drop=True)
L1, L2 = "eng", "spa"
data["y"] = (data["word_lang"] == L2).astype(int)

rng = np.random.default_rng(SEED)
dials = np.array(sorted(data["dialogue_id"].unique())); rng.shuffle(dials)
n = len(dials); n_tr, n_va = int(0.8*n), int(0.1*n)
tr_d, va_d = set(dials[:n_tr]), set(dials[n_tr:n_tr+n_va])
def split_name(d):
    if d in tr_d: return "train"
    if d in va_d: return "val"
    return "test"
data["split"] = data["dialogue_id"].map(split_name)
data = data.sort_values(["dialogue_id","sentence_id","word_index"]).reset_index(drop=True)

grp = data.groupby(["dialogue_id","sentence_id"], sort=False)
data["prev_y"]   = grp["y"].shift(1)
data["prev_pos"] = grp["pos_tag"].shift(1)
data["switch_true"] = (data["prev_y"].notna() &
                       (data["y"] != data["prev_y"])).astype("float")

print("Split sizes:", data.groupby("split").size().to_dict())
print("Overall within-sentence switch rate:",
      data.loc[data["prev_y"].notna(), "switch_true"].mean().round(4))

## 3. Train the LogReg baseline (features: prev-lang + prev-POS + speaker)

In [ ]:
base_df = data.dropna(subset=["prev_y","prev_pos"]).copy()
base_df["prev_y"] = base_df["prev_y"].astype(int)
tr = base_df[base_df["split"]=="train"]
te = base_df[base_df["split"]=="test"]

ohe_pos = OneHotEncoder(handle_unknown="ignore").fit(tr[["prev_pos"]])
ohe_spk = OneHotEncoder(handle_unknown="ignore").fit(tr[["speaker"]])

def featurize(d):
    return hstack([csr_matrix(d[["prev_y"]].values, dtype=float),
                   ohe_pos.transform(d[["prev_pos"]]),
                   ohe_spk.transform(d[["speaker"]])]).tocsr()

Xtr, Xte = featurize(tr), featurize(te)
ytr, yte = tr["y"].values, te["y"].values

t0 = time.time()
clf = LogisticRegression(max_iter=1000, n_jobs=-1).fit(Xtr, ytr)
print(f"LogReg fit in {time.time()-t0:.1f}s  ·  "
      f"test acc={clf.score(Xte, yte):.4f}")

# Place LogReg's per-token probabilities + hard preds back onto `data`.
data["lr_p"]    = np.nan
data["lr_pred"] = np.nan
data.loc[te.index, "lr_p"]    = clf.predict_proba(Xte)[:, 1]
data.loc[te.index, "lr_pred"] = (data.loc[te.index, "lr_p"] >= 0.5).astype(int)

## 4. Train the BiLSTM tagger

5 epochs is enough for val AUC to saturate. Set `EPOCHS = 8` if you want the
exact tutorial setting.


In [ ]:
PAD, UNK = "<pad>", "<unk>"
MIN_COUNT = 2
tr_words = data[data["split"]=="train"]
wc = Counter(tr_words["word"].tolist())
vocab = [PAD, UNK] + [w for w, c in wc.most_common() if c >= MIN_COUNT]
word2id = {w:i for i,w in enumerate(vocab)}
pos_vocab = [PAD] + sorted(tr_words["pos_tag"].unique().tolist())
pos2id = {p:i for i,p in enumerate(pos_vocab)}
enc_w = lambda w: word2id.get(w, word2id[UNK])
enc_p = lambda p: pos2id.get(p, 0)

def build_sents(df_):
    out = []
    for (d,s), g in df_.groupby(["dialogue_id","sentence_id"], sort=False):
        g = g.sort_values("word_index")
        out.append({"did": int(d), "sid": int(s),
                    "words": [enc_w(w) for w in g["word"]],
                    "pos":   [enc_p(p) for p in g["pos_tag"]],
                    "y":     g["y"].tolist()})
    return out

tr_sents = build_sents(data[data["split"]=="train"])
va_sents = build_sents(data[data["split"]=="val"])
te_sents = build_sents(data[data["split"]=="test"])

class DS(Dataset):
    def __init__(self, s): self.s = s
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        s = self.s[i]
        return (torch.tensor(s["words"], dtype=torch.long),
                torch.tensor(s["pos"],   dtype=torch.long),
                torch.tensor(s["y"],     dtype=torch.float))

def collate(batch):
    W = pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0)
    P = pad_sequence([b[1] for b in batch], batch_first=True, padding_value=0)
    Y = pad_sequence([b[2] for b in batch], batch_first=True, padding_value=0.0)
    M = (W != 0).float()
    return W, P, Y, M

BATCH = 32
tr_loader = DataLoader(DS(tr_sents), batch_size=BATCH, shuffle=True,  collate_fn=collate)
va_loader = DataLoader(DS(va_sents), batch_size=BATCH, shuffle=False, collate_fn=collate)
te_loader = DataLoader(DS(te_sents), batch_size=BATCH, shuffle=False, collate_fn=collate)

class BiLSTMTagger(nn.Module):
    def __init__(self, nw, np_, wd=64, pd_=16, h=64, do=0.3):
        super().__init__()
        self.we = nn.Embedding(nw, wd, padding_idx=0)
        self.pe = nn.Embedding(np_, pd_, padding_idx=0)
        self.lstm = nn.LSTM(wd+pd_, h, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(do)
        self.head = nn.Linear(2*h, 1)
    def forward(self, w, p):
        x = torch.cat([self.we(w), self.pe(p)], dim=-1)
        h, _ = self.lstm(x)
        return self.head(self.drop(h)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMTagger(len(word2id), len(pos2id)).to(device)
crit = nn.BCEWithLogitsLoss(reduction="none")
opt  = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss = 0.0; total_tok = 0.0
    all_p, all_y = [], []
    for w, p, y, m in loader:
        w, p, y, m = w.to(device), p.to(device), y.to(device), m.to(device)
        logits = model(w, p)
        lt = crit(logits, y) * m
        loss = lt.sum() / m.sum().clamp(min=1)
        if train_mode:
            opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()*m.sum().item(); total_tok += m.sum().item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        yy = y.cpu().numpy(); mm = m.cpu().numpy()
        for b in range(probs.shape[0]):
            L = int(mm[b].sum())
            all_p.append(probs[b, :L]); all_y.append(yy[b, :L])
    return total_loss/max(total_tok,1), np.concatenate(all_p), np.concatenate(all_y)

EPOCHS = 5
for ep in range(1, EPOCHS+1):
    tl, tp, ty = run_epoch(tr_loader, True)
    vl, vp, vy = run_epoch(va_loader, False)
    print(f"ep {ep}  train loss {tl:.4f} auc {roc_auc_score(ty,tp):.3f} | "
          f"val loss {vl:.4f} auc {roc_auc_score(vy,vp):.3f}")

### Place BiLSTM predictions back onto `data`

In [ ]:
model.eval()
bi_p    = np.full(len(data), np.nan)
bi_pred = np.full(len(data), np.nan)
with torch.no_grad():
    for s in te_sents:
        w = torch.tensor([s["words"]], dtype=torch.long).to(device)
        p = torch.tensor([s["pos"]],   dtype=torch.long).to(device)
        probs = torch.sigmoid(model(w, p)).cpu().numpy()[0]
        mask = ((data["dialogue_id"]==s["did"]) & (data["sentence_id"]==s["sid"]))
        idxs = sorted(data.index[mask].tolist(),
                      key=lambda i: data.at[i, "word_index"])
        for ii, prob in zip(idxs, probs):
            bi_p[ii]    = prob
            bi_pred[ii] = int(prob >= 0.5)
data["bi_p"]    = bi_p
data["bi_pred"] = bi_pred
print("done.")

## 5. Token-level metrics (reproduce the slide, for reference)

In [ ]:
for name, col in [("LogReg", "lr_pred"), ("BiLSTM", "bi_pred")]:
    m = (data["split"]=="test") & data[col].notna()
    yt, yh = data.loc[m,"y"].astype(int).values, data.loc[m,col].astype(int).values
    print(f"{name:6s} n={m.sum():>6d}  acc={accuracy_score(yt,yh):.4f}  "
          f"f1={f1_score(yt,yh):.4f}")

## 6. Switch-point evaluation

The *predicted* switch at position t is defined the same way as the true switch:
`pred_switch_t = 1  iff  pred_t ≠ pred_{t-1}` (within sentence).

For a continuous detection score we use `|p_t − p_{t−1}|` — how much the
model's language probability moved from the previous token to this one.
A real switch should cause a big jump; a continuation should not.

Eligibility filter: token is in test split, has a valid previous token, and
both the current and previous predictions exist (LogReg has no prediction for
the first word of every sentence).


In [ ]:
def predicted_switch_series(col):
    prev = data.groupby(["dialogue_id","sentence_id"], sort=False)[col].shift(1)
    return (prev.notna() & (data[col] != prev)).astype("float")

def score(name, pred_col, prob_col):
    sw_pred   = predicted_switch_series(pred_col)
    prev_prob = data.groupby(["dialogue_id","sentence_id"], sort=False)[prob_col].shift(1)
    switch_score = (data[prob_col] - prev_prob).abs()

    prev_pred = data.groupby(["dialogue_id","sentence_id"], sort=False)[pred_col].shift(1)
    mask = ((data["split"]=="test") & data["prev_y"].notna()
            & data[pred_col].notna() & prev_pred.notna())
    y_true = data.loc[mask,"switch_true"].astype(int).values
    y_pred = sw_pred.loc[mask].astype(int).values
    ss     = switch_score.loc[mask].values

    P,R,F,_ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {
        "model": name, "n": int(mask.sum()),
        "switch_rate": float(y_true.mean()),
        "precision": P, "recall": R, "f1": F,
        "auc_delta_p": roc_auc_score(y_true, ss),
        "ap_delta_p": average_precision_score(y_true, ss),
        "cm": confusion_matrix(y_true, y_pred),
    }

rows = [score("LogReg baseline", "lr_pred", "lr_p"),
        score("BiLSTM tagger",   "bi_pred", "bi_p")]
tbl = pd.DataFrame([{k:v for k,v in r.items() if k!='cm'} for r in rows]).round(3)
print(tbl.to_string(index=False))
for r in rows:
    print(f"\n{r['model']} confusion matrix (rows=truth, cols=pred; 0=no-switch, 1=switch):")
    print(r["cm"])

## 7. Interpreting the numbers

Compare to the headline token-level numbers from the slide:

| Metric           | LogReg | BiLSTM |
|------------------|-------:|-------:|
| **Token acc**    | 0.981  | 0.989  |
| **Token F1**     | 0.982  | 0.989  |
| **Switch F1**    | ~0.20  | ~0.64  |
| **Switch AP**    | ~0.06  | ~0.72  |

The 98%-ish token-level scores were carried almost entirely by the 98% of
tokens where language continues. On the ~2% of tokens where speakers actually
switch, LogReg is barely better than random (AP ≈ 0.06 on a ~1.8% base rate),
and BiLSTM is genuinely useful but far from saturated.

Takeaways for the project:

1. **Switch F1, not token accuracy, is the headline metric** for this
   problem — rewrite the slide accordingly.
2. **The context-aware BiLSTM gap gets *larger* on the real task**
   (Δ token-F1 ≈ 0.007 vs Δ switch-F1 ≈ 0.44), which is the right direction.
3. Next steps that should move switch-F1:
   - reformulate the task as switch detection end-to-end
     (`y_t = 1 iff lang_t ≠ lang_{t-1}`) and train on that,
   - add character-level features (spelling is a strong language cue),
   - use pretrained multilingual embeddings (fastText or XLM-R) instead of
     learning word embeddings from scratch.
